# CS:GO — Projeto completo (Google Colab)

Notebook **consolidado**: reúne todas as etapas em um único arquivo para **Runtime → Executar tudo**.

> **Setup:** a célula de `pip install` só adiciona pacotes do projeto (não força numpy/pandas). Use **Runtime → Executar tudo** uma vez.

## Mapa do notebook (palavras-chave)

| Parte | O quê | Palavras-chave |
|-------|--------|----------------|
| **00 — Setup** | Clone, `pip install`, download Kaggle | `kagglehub`, `requirements-colab.txt` |
| **01 — EDA** | Exploração visual dos dados | economia, buy tier, mapas, AWP, correlação |
| **02 — Pré-processamento** | Filtros, features, split 80/20 | `StandardScaler`, `OneHotEncoder`, `target_cls` |
| **03 — Modelagem** | Treino e comparação de modelos | ver tabela abaixo |

### Parte 03 — onde fica cada coisa

| Seção | Modelos / tarefa | Métricas |
|-------|------------------|----------|
| **03.1 Classificação** | Regressão Logística, **Random Forest**, SVM, **KNN** | accuracy, **F1**, **ROC-AUC** |
| **03.2 Regressão** | Regressão Linear, Random Forest Regressor | **MAE**, **RMSE**, **R²** |
| **03.3 Métricas** | Gráficos + JSON final | `model_results.json`, curva ROC, matriz de confusão |

**Alvo classificação:** `target_cls` → CT (0) ou T (1) vence o round?  
**Alvo regressão:** `money_diff` → diferença de dinheiro CT − T (valor contínuo em $).

Os notebooks `00_`–`03_` continuam separados.

| Parte | Status |
|-------|--------|
| 00 — Setup | ✅ |
| 01 — EDA | ✅ |
| 02 — Pré-processamento | ✅ |
| 03 — Modelagem | ✅ |

---
# PARTE 00 — Setup

Espelha `notebooks/00_Setup_Colab.ipynb`

## 00.1 Clonar repositório

In [ ]:
from pathlib import Path

# Pasta do projeto no ambiente Colab
REPO_DIR = Path("projeto-machine-learning-csgo")
if not REPO_DIR.exists():
    # Primeira execução: clona o repositório do GitHub (notebooks + requirements)
    !git clone https://github.com/carloseduardob/projeto-machine-learning-csgo.git
# Passa a trabalhar dentro do repo clonado
%cd projeto-machine-learning-csgo

## 00.2 Instalar dependências

In [ ]:
import subprocess
import sys

# Instala só o que falta (kagglehub, seaborn, etc.) — sem forçar reinstall do pandas do Colab
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)

import numpy as np
import pandas as pd
import sklearn

# Confere versões das bibliotecas principais
print(f"numpy {np.__version__} | pandas {pd.__version__} | sklearn {sklearn.__version__}")

## 00.3 Download do dataset (kagglehub)

Dataset público — não exige credencial Kaggle.

In [ ]:
import shutil

import kagglehub
import pandas as pd

# ID do dataset no Kaggle (público — não precisa de API key)
DATASET = "christianlillelund/csgo-round-winner-classification"
DATA_RAW = Path("/content/data/raw")  # onde salvamos o CSV no Colab
DATA_RAW.mkdir(parents=True, exist_ok=True)

# kagglehub baixa para cache local e devolve o caminho
cache_path = Path(kagglehub.dataset_download(DATASET))
print("Cache kagglehub:", cache_path)

csv_path = None
for src in cache_path.rglob("*.csv"):
    dest = DATA_RAW / src.name
    shutil.copy2(src, dest)  # copia cada CSV para /content/data/raw/
    print("Copiado →", dest)
    if src.name == "csgo_round_snapshots.csv" or csv_path is None:
        csv_path = dest  # arquivo principal do projeto

if csv_path is None:
    raise FileNotFoundError(f"Nenhum CSV em {cache_path}")

CSV_PATH = csv_path  # variável usada nas Partes 01 e 02
print("Arquivo principal:", CSV_PATH)

## 00.4 Carregar e inspecionar

In [ ]:
# Lê o CSV inteiro para um DataFrame (tabela pandas)
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)  # esperado: (122410, 97) — linhas × colunas
print("Nulos:", df.isna().sum().sum())  # 0 = dataset limpo, sem valores ausentes
print("\nColunas (primeiras 15):", list(df.columns[:15]))
df.head()  # primeiras 5 linhas — inspeção visual rápida

In [ ]:
# Balanceamento das classes: quantos snapshots terminam com CT vs T vencedor
print("Vencedor do round:")
print(df["round_winner"].value_counts())  # ~51% T / ~49% CT

print("\nMapas:")
print(df["map"].value_counts().head())  # quais mapas aparecem mais no dataset

---
# PARTE 01 — EDA

Espelha `notebooks/01_EDA.ipynb`

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Estilo dos gráficos da EDA
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.bbox"] = "tight"

FIGURES = Path("/content/reports/figures")  # pasta onde os PNGs serão salvos
FIGURES.mkdir(parents=True, exist_ok=True)

csv_path = CSV_PATH  # reutiliza o caminho definido na Parte 00
print("Usando:", csv_path)

In [ ]:
BUY_TIER_ORDER = ["force", "eco", "mixed", "half", "full"]  # ordem lógica no gráfico de buy tier


def buy_tier(team_money: float) -> str:
    """Classifica a fase de compra do time pela média de $ por jogador (team_money / 5)."""
    m = team_money / 5
    if m >= 4700:
        return "full"
    if 2800 <= m < 3800:
        return "half"
    if m < 1700:
        return "force"
    if m < 2000:
        return "eco"
    return "mixed"


df = pd.read_csv(csv_path)

# --- Features derivadas para a EDA (explorar antes de modelar) ---
df["money_diff"] = df["ct_money"] - df["t_money"]  # vantagem econômica CT − T em $
df["health_diff"] = df["ct_health"] - df["t_health"]
df["armor_diff"] = df["ct_armor"] - df["t_armor"]
df["players_alive_diff"] = df["ct_players_alive"] - df["t_players_alive"]
df["ct_richer"] = df["money_diff"] > 0  # True se CT tem mais dinheiro
df["richer_wins"] = (
    ((df["round_winner"] == "CT") & df["ct_richer"])
    | ((df["round_winner"] == "T") & ~df["ct_richer"])
)  # heurística: "time com mais $ venceu este snapshot?"
df["target"] = (df["round_winner"] == "T").astype(int)  # 1 = T vence, 0 = CT vence
df["ct_buy_tier"] = df["ct_money"].map(buy_tier)  # eco, force, half, full...
df["t_buy_tier"] = df["t_money"].map(buy_tier)

print("Shape:", df.shape)
print("Nulos:", df.isna().sum().sum())
print("\nVencedor:")
print(df["round_winner"].value_counts())
print(f"\nTime com mais $ vence: {df['richer_wins'].mean():.1%}")  # ~60,5% — regra simples falha
print(f"money_diff médio (CT−T): {df['money_diff'].mean():.0f} USD")

## 01.1 Balanceamento CT vs T

In [ ]:
# Gráfico 01 — balanceamento CT vs T (classes equilibradas ~51/49)
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["round_winner"].value_counts()
sns.barplot(x=counts.index, y=counts.values, ax=ax, hue=counts.index, legend=False)
ax.set_title("Distribuição do vencedor do round")
ax.set_xlabel("Lado vencedor")
ax.set_ylabel("Quantidade de snapshots")
for i, v in enumerate(counts.values):
    ax.text(i, v + 500, f"{v:,}", ha="center", fontsize=9)
plt.show()
fig.savefig(FIGURES / "01_distribuicao_vencedor.png")
plt.close(fig)

## 01.2 Economia

In [ ]:
# Gráfico 02 — economia: distribuição de $ por lado + money_diff por vencedor
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["ct_money"], label="CT", ax=axes[0], kde=True, stat="density", alpha=0.6)
sns.histplot(df["t_money"], label="T", ax=axes[0], kde=True, stat="density", alpha=0.6)
axes[0].set_title("Economia total por lado (USD)")
axes[0].legend()
sns.boxplot(data=df, x="round_winner", y="money_diff", hue="round_winner", ax=axes[1], legend=False)
axes[1].axhline(0, color="gray", ls="--", lw=0.8)  # linha em zero = economia igual
axes[1].set_title("Diferença de economia (CT − T) por vencedor")
plt.tight_layout()
plt.show()
fig.savefig(FIGURES / "02_economia.png")
plt.close(fig)

## 01.3 Mapas

In [ ]:
# Gráfico 03 — quantos snapshots por mapa + taxa de vitória T em cada mapa
map_counts = df["map"].value_counts()
map_win = df.groupby("map")["target"].mean().sort_values(ascending=False)  # proporção vitórias T

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=map_counts.index, y=map_counts.values, ax=axes[0], hue=map_counts.index, legend=False)
axes[0].set_title("Snapshots por mapa")
axes[0].tick_params(axis="x", rotation=30)
sns.barplot(x=map_win.index, y=map_win.values, ax=axes[1], hue=map_win.index, legend=False)
axes[1].axhline(0.5, color="gray", ls="--", lw=0.8)
axes[1].set_title("Taxa de vitória T por mapa")
axes[1].set_ylabel("Proporção vitórias T")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()
fig.savefig(FIGURES / "03_mapas.png")
plt.close(fig)

## 01.4 Buy tier

Faixas por média de dinheiro por jogador (`team_money / 5`). Categoria **mixed** nas transições entre tiers.

In [ ]:
# Gráfico 04 — taxa de vitória por buy tier (eco, force, half, full...)
ct_rates = df.groupby("ct_buy_tier")["target"].apply(lambda s: 1 - s.mean())  # taxa CT vence
t_rates = df.groupby("t_buy_tier")["target"].mean()  # taxa T vence
order = BUY_TIER_ORDER

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(order))
width = 0.35
ax.bar(x - width / 2, [ct_rates.get(t, 0) for t in order], width, label="CT vence")
ax.bar(x + width / 2, [t_rates.get(t, 0) for t in order], width, label="T vence")
ax.set_xticks(x)
ax.set_xticklabels(order)
ax.set_title("Taxa de vitória por tier de compra")
ax.set_ylabel("Proporção de vitórias")
ax.legend()
plt.show()
fig.savefig(FIGURES / "04_buy_tier.png")
plt.close(fig)

## 01.5 Armamento e AWP

In [ ]:
# Gráfico 05 — top 10 armas mais frequentes nos snapshots (CT vs T)
weapons = [c for c in df.columns if c.startswith(("ct_weapon_", "t_weapon_"))]
ct_weapons = [c for c in weapons if c.startswith("ct_weapon_")]
ct_totals = df[ct_weapons].sum().sort_values(ascending=False).head(10)
t_weapons = [c.replace("ct_", "t_") for c in ct_totals.index]
t_totals = df[t_weapons].sum().reindex(t_weapons)
labels = [c.replace("ct_weapon_", "").upper() for c in ct_totals.index]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(labels))
width = 0.35
ax.bar(x - width / 2, ct_totals.values, width, label="CT")
ax.bar(x + width / 2, t_totals.values, width, label="T")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=35, ha="right")
ax.set_title("Top 10 armas nos snapshots")
ax.set_ylabel("Soma de contagem")
ax.legend()
plt.show()
fig.savefig(FIGURES / "05_armas_top10.png")
plt.close(fig)

# Gráfico 06 — ter AWP no time muda a taxa de vitória T?
awp_ct = df.groupby(df["ct_weapon_awp"] > 0)["target"].apply(lambda s: 1 - s.mean())
awp_t = df.groupby(df["t_weapon_awp"] > 0)["target"].mean()
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(
    x=["CT sem AWP", "CT com AWP", "T sem AWP", "T com AWP"],
    y=[awp_ct.get(False, np.nan), awp_ct.get(True, np.nan), awp_t.get(False, np.nan), awp_t.get(True, np.nan)],
    hue=["CT sem AWP", "CT com AWP", "T sem AWP", "T com AWP"],
    ax=ax,
    legend=False,
)
ax.set_title("Taxa de vitória T vs presença de AWP")
ax.set_ylabel("Proporção vitórias T")
ax.tick_params(axis="x", rotation=25)
plt.show()
fig.savefig(FIGURES / "06_awp_impacto.png")
plt.close(fig)

## 01.6 Bomba plantada

In [ ]:
# Gráfico 07 — com bomba plantada, T tende a vencer mais (objetivo do ataque)
rates = df.groupby("bomb_planted")["target"].mean()
fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(
    x=["Não plantada", "Plantada"],
    y=[rates.get(False, 0), rates.get(True, 0)],
    hue=["Não plantada", "Plantada"],
    ax=ax,
    legend=False,
)
ax.axhline(0.5, color="gray", ls="--", lw=0.8)
ax.set_title("Taxa de vitória T vs bomba plantada")
ax.set_ylabel("Proporção vitórias T")
plt.show()
fig.savefig(FIGURES / "07_bomba_plantada.png")
plt.close(fig)

## 01.7 Tempo restante e jogadores vivos

In [ ]:
# Gráfico 08 — tempo restante no round + relação jogadores vivos vs economia
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["time_left"], bins=40, ax=axes[0], kde=True)
axes[0].set_title("Tempo restante no round (segundos)")  # vários snapshots/round → risco de leakage
sample = df.sample(min(8000, len(df)), random_state=42)  # amostra para scatter legível
sns.scatterplot(
    data=sample,
    x="players_alive_diff",
    y="money_diff",
    hue="round_winner",
    alpha=0.35,
    ax=axes[1],
)
axes[1].set_title("Jogadores vivos (CT−T) vs economia (amostra)")
plt.tight_layout()
plt.show()
fig.savefig(FIGURES / "08_tempo_jogadores_vivos.png")
plt.close(fig)

## 01.8 Correlações

In [ ]:
# Gráfico 09 — heatmap de correlação entre variáveis-chave
cols = [
    "time_left",
    "money_diff",
    "health_diff",
    "armor_diff",
    "players_alive_diff",
    "bomb_planted",
    "target",
]
corr = df[cols].copy()
corr["bomb_planted"] = corr["bomb_planted"].astype(int)  # bool → 0/1 para correlacionar

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlação — variáveis-chave")
plt.show()
fig.savefig(FIGURES / "09_correlacao.png")
plt.close(fig)

### Leakage temporal (`time_left`)

Snapshots no fim do round podem vazar o resultado. **Decisão:** filtrar `time_left >= 150` no pré-processamento.

## 01.8b Outliers (IQR)

Quantificação de valores extremos nas variáveis econômicas. **Decisão:** manter outliers — rounds com economia muito desbalanceada trazem informação tática relevante.

In [ ]:
def iqr_outliers(series: pd.Series) -> dict:
    """Conta outliers pela regra IQR: fora de [Q1−1.5·IQR, Q3+1.5·IQR]."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (series < low) | (series > high)
    return {
        "count": int(mask.sum()),
        "pct": round(mask.mean() * 100, 2),
        "low": round(float(low), 0),
        "high": round(float(high), 0),
    }

outlier_cols = ["money_diff", "ct_money", "t_money"]
outlier_stats = {col: iqr_outliers(df[col]) for col in outlier_cols}
print(json.dumps(outlier_stats, indent=2, ensure_ascii=False))  # ~8% em money_diff

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df[outlier_cols], orient="h", ax=ax)
ax.set_title("Outliers — variáveis econômicas (regra IQR)")
plt.tight_layout()
plt.show()
fig.savefig(FIGURES / "08b_outliers_economia.png")
plt.close(fig)

## 01.9 Resumo numérico

In [ ]:
# Resumo numérico da EDA — salvo em JSON para o relatório
summary = {
    "rows": len(df),
    "columns": len(df.columns),
    "nulls_total": int(df.isna().sum().sum()),
    "winner_counts": df["round_winner"].value_counts().to_dict(),
    "winner_pct_t": round(df["target"].mean() * 100, 2),
    "richer_wins_pct": round(df["richer_wins"].mean() * 100, 2),
    "maps": df["map"].value_counts().to_dict(),
    "time_left_mean": round(df["time_left"].mean(), 2),
    "time_left_median": round(df["time_left"].median(), 2),
    "money_diff_mean": round(df["money_diff"].mean(), 2),
    "bomb_planted_pct": round(df["bomb_planted"].mean() * 100, 2),
    "awp_ct_pct": round((df["ct_weapon_awp"] > 0).mean() * 100, 2),
    "awp_t_pct": round((df["t_weapon_awp"] > 0).mean() * 100, 2),
    "outliers_iqr": outlier_stats,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
(FIGURES / "eda_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"\nGráficos salvos em: {FIGURES}")

---
# PARTE 02 — Pré-processamento

Espelha `notebooks/02_Preprocessamento.ipynb`

In [ ]:
# --- Configuração da Parte 02: pré-processamento ---

MAP_EXCLUDE = {"de_cache"}  # mapa com pouquíssimos dados — removemos
RIFLE_COLS_CT = [
    "ct_weapon_ak47", "ct_weapon_m4a4", "ct_weapon_m4a1s",
    "ct_weapon_galilar", "ct_weapon_famas", "ct_weapon_sg553", "ct_weapon_aug",
]
RIFLE_COLS_T = [c.replace("ct_", "t_") for c in RIFLE_COLS_CT]  # mesmas armas, lado T
UTIL_COLS_CT = [
    "ct_grenade_hegrenade", "ct_grenade_flashbang", "ct_grenade_smokegrenade",
    "ct_grenade_incendiarygrenade", "ct_grenade_molotovgrenade", "ct_grenade_decoygrenade",
]
UTIL_COLS_T = [c.replace("ct_", "t_") for c in UTIL_COLS_CT]

BUY_TIER_ORDER = ["force", "eco", "mixed", "half", "full"]


def buy_tier(team_money: float) -> str:
    """Mesma lógica da EDA: categoriza compra pelo $ médio por jogador."""
    m = team_money / 5
    if m >= 4700:
        return "full"
    if 2800 <= m < 3800:
        return "half"
    if m < 1700:
        return "force"
    if m < 2000:
        return "eco"
    return "mixed"


def filter_round_start(df: pd.DataFrame) -> pd.DataFrame:
    """Mantém só snapshots do INÍCIO do round, 5v5, sem mapa raro — anti-leakage."""
    mask = df["time_left"] >= 150  # início do round; evita snapshots do fim (leakage)
    mask &= df["ct_players_alive"] == 5  # todos CT vivos
    mask &= df["t_players_alive"] == 5  # todos T vivos
    mask &= ~df["map"].isin(MAP_EXCLUDE)  # exclui de_cache
    return df.loc[mask].copy()  # só linhas que passam em TODAS as condições


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """Cria colunas novas (feature engineering) a partir das originais."""
    out = df.copy()
    new_cols = pd.DataFrame(
        {
            "money_diff": out["ct_money"] - out["t_money"],  # diferença em $ (CT − T)
            "health_diff": out["ct_health"] - out["t_health"],
            "armor_diff": out["ct_armor"] - out["t_armor"],
            "players_alive_diff": out["ct_players_alive"] - out["t_players_alive"],
            "money_ratio": out["ct_money"] / out["t_money"].replace(0, 1),  # razão CT/T; replace evita ÷0
            "ct_has_awp": (out["ct_weapon_awp"] > 0).astype(int),  # 1 se time CT tem AWP
            "t_has_awp": (out["t_weapon_awp"] > 0).astype(int),
            "ct_rifle_count": out[RIFLE_COLS_CT].sum(axis=1),  # soma rifles CT nesta linha
            "t_rifle_count": out[RIFLE_COLS_T].sum(axis=1),
            "ct_util_count": out[UTIL_COLS_CT].sum(axis=1),  # granadas/utilitários
            "t_util_count": out[UTIL_COLS_T].sum(axis=1),
            "ct_buy_tier": out["ct_money"].map(buy_tier),
            "t_buy_tier": out["t_money"].map(buy_tier),
            "target_cls": (out["round_winner"] == "T").astype(int),  # alvo: 1=T vence, 0=CT
        }
    )
    return pd.concat([out, new_cols], axis=1)  # junta colunas novas ao DataFrame original


def get_feature_columns(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    """Separa colunas numéricas vs categóricas para o pipeline sklearn."""
    numeric_base = [
        "time_left", "ct_score", "t_score", "ct_health", "t_health",
        "ct_armor", "t_armor", "ct_money", "t_money", "ct_helmets", "t_helmets",
        "ct_defuse_kits", "ct_players_alive", "t_players_alive",
    ]
    derived = [
        "money_diff", "health_diff", "armor_diff", "players_alive_diff",
        "money_ratio", "ct_has_awp", "t_has_awp", "ct_rifle_count",
        "t_rifle_count", "ct_util_count", "t_util_count",
    ]
    weapon_cols = [c for c in df.columns if c.startswith(("ct_weapon_", "t_weapon_"))]
    grenade_cols = [c for c in df.columns if c.startswith(("ct_grenade_", "t_grenade_"))]
    numeric = numeric_base + derived + weapon_cols + grenade_cols + ["bomb_planted"]
    numeric = [c for c in numeric if c in df.columns]
    categorical = [c for c in ["map", "ct_buy_tier", "t_buy_tier"] if c in df.columns]
    return numeric, categorical


from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PROCESSED = Path("/content/data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(CSV_PATH)  # 122.410 linhas — dataset completo
print("Bruto:", df_raw.shape)
df = add_derived_features(filter_round_start(df_raw))  # filtros + features → ~33.365 linhas
print("Após filtros:", df.shape)
print(f"target_cls (T): {df['target_cls'].mean():.1%}")  # balanceamento ~51% T

df.to_csv(DATA_PROCESSED / "rounds_start.csv", index=False)  # base limpa para modelagem
meta = {
    "rows": len(df),
    "target_cls_balance_t_pct": round(df["target_cls"].mean() * 100, 2),
}
(DATA_PROCESSED / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(json.dumps(meta, indent=2))

In [ ]:
def make_preprocessor(df: pd.DataFrame, exclude: set = None) -> ColumnTransformer:
    """Pipeline de transformação: escala numéricas + one-hot em categorias."""
    numeric, categorical = get_feature_columns(df)

    if exclude is not None:
        numeric = [col for col in numeric if col not in exclude]
        categorical = [col for col in categorical if col not in exclude]

    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric),  # z = (x − μ) / σ — importante para KNN/SVM
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ],
        remainder="drop",
    )

numeric, categorical = get_feature_columns(df)
feature_cols = numeric + categorical
X_cls, y_cls = df[feature_cols], df["target_cls"]  # X = entradas, y = quem vence (0/1)

# Split 80% treino / 20% teste — estratificado mantém % CT/T igual nos dois conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

REGRESSION_EXCLUDE = {"money_diff", "money_ratio", "ct_money", "t_money"}
reg_cols = [c for c in feature_cols if c not in REGRESSION_EXCLUDE]
# Regressão prevê money_diff — não pode usar colunas de dinheiro na entrada (leakage)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    df[reg_cols], df["money_diff"], test_size=0.2, random_state=42
)

preprocessor = make_preprocessor(df)
X_train_t = preprocessor.fit_transform(X_train)  # TREINO: aprende média/desvio e transforma
X_test_t = preprocessor.transform(X_test)  # TESTE: só aplica (não vaza info do teste)
print("Classificação — treino:", X_train.shape, "| pipeline:", X_train_t.shape)
print("Regressão — treino:", X_train_r.shape)
# A Parte 03 usa esses splits para treinar LR, RF, SVM e KNN

---
# PARTE 03 — Modelagem

Espelha `notebooks/03_Modelagem.ipynb`. Usa `df` e splits da Parte 02.

Temos **duas tarefas** de Machine Learning:

1. **Classificação** — prever o **vencedor** do round (CT ou T)
2. **Regressão** — prever a **diferença de dinheiro** (`money_diff`)

Todos os modelos passam pelo mesmo **pipeline**: `StandardScaler` (numéricas) + `OneHotEncoder` (mapa, buy tier) → algoritmo.
Hiperparâmetros são escolhidos com **GridSearchCV** (validação cruzada 3-fold).

In [ ]:
import json
import os
import time
from pathlib import Path

# Colab: limitar threads evita OSError do threadpoolctl (dlopen) com joblib paralelo
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
N_JOBS = 2  # no Colab, n_jobs=-1 conflita com threadpoolctl; use 2 workers

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# =============================================================================
# 03.1 — CLASSIFICAÇÃO
# Pergunta: quem vence o round?  Alvo: target_cls (0=CT, 1=T)
# Métricas avaliadas: accuracy, precision, recall, F1 (GridSearchCV), ROC-AUC
# =============================================================================

RANDOM_STATE = 42
SVM_MAX_SAMPLES = 6_000  # SVM é pesado — treina só em 6k linhas estratificadas
REGRESSION_EXCLUDE = {"money_diff", "money_ratio", "ct_money", "t_money"}
METRICS = Path("/content/reports/metrics")   # pasta das métricas finais (JSON)
FIGURES = Path("/content/reports/figures")   # pasta dos gráficos exportados
METRICS.mkdir(parents=True, exist_ok=True)

preprocessor_cls = make_preprocessor(df)
preprocessor_reg = make_preprocessor(df, exclude=REGRESSION_EXCLUDE)

def _roc_auc(m, X, y):
    s = m.predict_proba(X)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(X)
    return float(roc_auc_score(y, s))

def _sub(X, y, n):
    if len(X) <= n:
        return X, y
    X_sub, _, y_sub, _ = train_test_split(X, y, train_size=n, stratify=y, random_state=RANDOM_STATE)
    return X_sub, y_sub

X_svm, y_svm = _sub(X_train, y_train, SVM_MAX_SAMPLES)

# cls_cfgs: (nome, estimador, grid de hiperparâmetros, X_treino, y_treino)
#   logistic_regression — Regressão Logística (baseline interpretável)
#   random_forest       — Random Forest / Floresta Aleatória (melhor F1 no projeto)
#   svm_rbf             — SVM com kernel RBF (fronteira não linear)
#   knn                 — KNN / K-Nearest Neighbors (Aula 5)
#
# KNN: não aprende fórmula — busca os K rounds de treino mais parecidos (distância
#      euclidiana após StandardScaler) e vota na classe majoritária (CT ou T).
#      GridSearch testa n_neighbors em {5, 11, 21}.
cls_cfgs = [
    ("logistic_regression", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, class_weight="balanced"), {"clf__C": [0.1, 1.0, 10.0]}, X_train, y_train),
    ("random_forest", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=N_JOBS), {"clf__max_depth": [12, 20], "clf__min_samples_leaf": [1, 5]}, X_train, y_train),
    ("svm_rbf", SVC(kernel="rbf", random_state=RANDOM_STATE, class_weight="balanced"), {"clf__C": [0.5, 1.0], "clf__gamma": ["scale"]}, X_svm, y_svm),
    ("knn", KNeighborsClassifier(), {"clf__n_neighbors": [5, 11, 21]}, X_train, y_train),
]
cls_results, fitted_cls, best_name, best_f1 = {}, {}, None, -1.0
for name, est, grid_p, Xf, yf in cls_cfgs:
    print(f"Classificação — {name}...")
    pipe = Pipeline([("prep", preprocessor_cls), ("clf", est)])
    g = GridSearchCV(pipe, grid_p, cv=3, scoring="f1", n_jobs=N_JOBS).fit(Xf, yf)
    m = g.best_estimator_
    fitted_cls[name] = m
    pred = m.predict(X_test)
    cls_results[name] = {
        "best_params": g.best_params_,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred), 4),
        "recall": round(recall_score(y_test, pred), 4),
        "f1": round(f1_score(y_test, pred), 4),
        "roc_auc": round(_roc_auc(m, X_test, y_test), 4),
        "train_samples": len(Xf),
    }
    if cls_results[name]["f1"] > best_f1:
        best_f1, best_name = cls_results[name]["f1"], name
cls_results["best_model"] = best_name
print("Melhor classificador:", best_name, "| F1 =", best_f1)
print("\nClassification report —", best_name)
print(classification_report(y_test, fitted_cls[best_name].predict(X_test), target_names=["CT", "T"]))
pd.DataFrame(cls_results).T.drop(index="best_model", errors="ignore")

## 03.1 — Classificação (CT vs T)

**Alvo:** `target_cls` (0 = CT vence, 1 = T vence)

| Modelo | Classe sklearn | Onde no código |
|--------|----------------|----------------|
| Regressão Logística | `LogisticRegression` | `cls_cfgs[0]` |
| Random Forest | `RandomForestClassifier` | `cls_cfgs[1]` — **melhor modelo** |
| SVM (RBF) | `SVC` | `cls_cfgs[2]` — subamostra 6k |
| **KNN** | `KNeighborsClassifier` | `cls_cfgs[3]` — Aula 5 |

**Métricas:** accuracy, **precision**, **recall**, **F1** (GridSearchCV), **ROC-AUC**

In [ ]:
# =============================================================================
# 03.2 — REGRESSÃO
# Pergunta: qual a diferença de dinheiro CT − T?  Alvo: money_diff (contínuo, $)
# Métricas: MAE (erro médio $), RMSE (penaliza erros grandes), R² (variância explicada)
# REGRESSION_EXCLUDE: remove dinheiro bruto e derivados diretos (evita leakage no alvo)
# =============================================================================
#
# Tipos de regressão comparados:
#   linear_regression       — Regressão Linear (baseline, relação linear)
#   random_forest_regressor — Random Forest Regressor (captura não linearidade)
#
reg_cfgs = [
    ("linear_regression", LinearRegression(), {}, False),
    ("random_forest_regressor", RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=N_JOBS), {"reg__max_depth": [12, 20], "reg__min_samples_leaf": [1, 5]}, True),
]
reg_results, fitted_reg, best_reg, best_rmse = {}, {}, None, float("inf")
for name, est, grid_p, use_grid in reg_cfgs:
    print(f"Regressão — {name}...")
    pipe = Pipeline([("prep", preprocessor_reg), ("reg", est)])
    if use_grid:
        g = GridSearchCV(pipe, grid_p, cv=3, scoring="neg_root_mean_squared_error", n_jobs=N_JOBS).fit(X_train_r, y_train_r)
        model, bp = g.best_estimator_, g.best_params_
    else:
        pipe.fit(X_train_r, y_train_r)
        model, bp = pipe, {}
    fitted_reg[name] = model
    pred = model.predict(X_test_r)
    rmse = float(np.sqrt(mean_squared_error(y_test_r, pred)))
    reg_results[name] = {"rmse": round(rmse, 2), "mae": round(mean_absolute_error(y_test_r, pred), 2), "r2": round(r2_score(y_test_r, pred), 4), "best_params": bp}
    if rmse < best_rmse:
        best_rmse, best_reg = rmse, name
reg_results["best_model"] = best_reg
print("Melhor regressão:", best_reg, "| RMSE =", best_rmse)
pd.DataFrame(reg_results).T.drop(index="best_model", errors="ignore")

## 03.2 — Regressão (`money_diff`)

**Alvo:** `money_diff` = dinheiro CT − dinheiro T (valor contínuo em $)

| Modelo | Classe sklearn | Onde no código |
|--------|----------------|----------------|
| Regressão Linear | `LinearRegression` | `reg_cfgs[0]` — baseline |
| Random Forest Regressor | `RandomForestRegressor` | `reg_cfgs[1]` — melhor R² |

**Métricas:** **MAE**, **RMSE**, **R²**

`REGRESSION_EXCLUDE` remove `ct_money`, `t_money`, `money_diff` e `money_ratio` para não vazar o alvo.

In [ ]:
# =============================================================================
# 03.3 — MÉTRICAS E GRÁFICOS
#   FIGURES/10_roc_classificacao.png, 11_matriz_confusao.png
#   FIGURES/12_importancia_classificacao.png, 13_regressao_real_vs_previsto.png
#   FIGURES/14_importancia_regressao.png
#   METRICS/model_results.json
# =============================================================================

fig, ax = plt.subplots(figsize=(7, 5))
for name, model in fitted_cls.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name.replace("_", " "))
ax.set_title("Curvas ROC — classificação")
plt.show()
fig.savefig(FIGURES / "10_roc_classificacao.png")
plt.close(fig)

best_cls = fitted_cls[best_name]
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(best_cls, X_test, y_test, ax=ax, cmap="Blues")
ax.set_title(f"Matriz de confusão — {best_name}")
plt.show()
fig.savefig(FIGURES / "11_matriz_confusao.png")
plt.close(fig)

if "random_forest" in fitted_cls:
    rf = fitted_cls["random_forest"]
    names = rf.named_steps["prep"].get_feature_names_out()
    imp = rf.named_steps["clf"].feature_importances_
    idx = np.argsort(imp)[-15:]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(names[idx], imp[idx])
    ax.set_title("Top 15 — importância (RF classificação)")
    plt.tight_layout()
    plt.show()
    fig.savefig(FIGURES / "12_importancia_classificacao.png")
    plt.close(fig)

best_reg_model = fitted_reg[best_reg]
y_pred_r = best_reg_model.predict(X_test_r)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_test_r, y_pred_r, alpha=0.2, s=8)
lims = [min(y_test_r.min(), y_pred_r.min()), max(y_test_r.max(), y_pred_r.max())]
ax.plot(lims, lims, "r--", lw=1)
ax.set_xlabel("money_diff real")
ax.set_ylabel("money_diff previsto")
ax.set_title(f"Regressão — {best_reg}")
plt.show()
fig.savefig(FIGURES / "13_regressao_real_vs_previsto.png")
plt.close(fig)

if "random_forest_regressor" in fitted_reg:
    rf_reg = fitted_reg["random_forest_regressor"]
    names = rf_reg.named_steps["prep"].get_feature_names_out()
    imp = rf_reg.named_steps["reg"].feature_importances_
    idx = np.argsort(imp)[-15:]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(names[idx], imp[idx])
    ax.set_title("Top 15 — importância (RF regressão)")
    plt.tight_layout()
    plt.show()
    fig.savefig(FIGURES / "14_importancia_regressao.png")
    plt.close(fig)

summary = {
    "rows_used": len(df),
    "svm_max_train_samples": SVM_MAX_SAMPLES,
    "classification": cls_results,
    "regression": reg_results,
}
(METRICS / "model_results.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("Concluído. Gráficos:", FIGURES)
print("Métricas:", METRICS / "model_results.json")

## 03.3 — Métricas e gráficos finais

| Saída | Caminho | Conteúdo |
|-------|---------|----------|
| Curva ROC | `/content/reports/figures/10_roc_classificacao.png` | Compara LR, RF, SVM, KNN |
| Matriz de confusão | `/content/reports/figures/11_matriz_confusao.png` | Melhor classificador |
| Feature importance | `/content/reports/figures/12_importancia_classificacao.png` | Top 15 atributos (RF) |
| Regressão real vs previsto | `/content/reports/figures/13_regressao_real_vs_previsto.png` | Melhor regressão |
| Feature importance (reg.) | `/content/reports/figures/14_importancia_regressao.png` | Top 15 atributos (RF) |
| **Métricas JSON** | `/content/reports/metrics/model_results.json` | Precision, Recall, F1, ROC-AUC, MAE, RMSE, R² |

O JSON reúne os resultados de **classificação** e **regressão** para o relatório e a apresentação.